# Fine-Tuning Gemma 2 2B → Apex AI (modelo PARTICULAR, portável, sem vendor lock-in)

Este notebook treina o **Gemma 2 2B IT** com o dataset da **Apex AI** (500+ exemplos) e
exporta um modelo **GGUF portável** que roda **localmente** — no Apex Desktop (`.exe`), no
**site** (apexglobalai.com) e nos **apps de celular** — via **Ollama / llama.cpp**, sem
depender de HuggingFace, Vertex ou nenhum provedor pago por token.

> ✅ **Sem token, sem login, sem lock-in.** A base do Gemma é baixada de um espelho
> **aberto** (não exige token do Hugging Face). O modelo final é 100% seu e roda offline.

**Fluxo:**
1. Carrega o dataset da Apex (`apex_train.jsonl` = treino, `apex_test.jsonl` = teste separado)
2. Fine-tuning com **LoRA** (eficiente, roda na GPU T4 grátis do Colab)
3. Funde os pesos LoRA no modelo base
4. Converte para **GGUF** (formato portável do llama.cpp)
5. **Avalia no conjunto de teste** (perguntas que o modelo NÃO viu no treino)
6. Baixa o `.gguf` + gera o `Modelfile` do Ollama para uso local

**Tempo estimado:** 40-70 min com GPU T4 (grátis).

## Instruções:
1. **Runtime → Change runtime type → T4 GPU**
2. Execute as células em ordem (ou Ctrl+F9 para tudo)
3. Ao final: baixe `apex-ai.gguf` e rode local com Ollama — **o modelo é seu**.


## 1. Instalar Dependências

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1) GPU obrigatória — se der "False", ative: Runtime → Change runtime type → T4 GPU
# ═══════════════════════════════════════════════════════════
import torch
if not torch.cuda.is_available():
    raise SystemError(
        "❌ SEM GPU! Ative agora: menu Runtime → Change runtime type → Hardware accelerator = T4 GPU, "
        "depois rode esta célula de novo. O fine-tuning NÃO funciona sem GPU."
    )
print(f"✅ GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ═══════════════════════════════════════════════════════════
# 2) Dependências — versões recentes e compatíveis (evita conflito de numpy)
# ═══════════════════════════════════════════════════════════
# Não fixamos versões antigas: deixamos o pip resolver versões recentes que já
# suportam numpy 2.x do Colab. Isso elimina os avisos de "numpy incompatible".
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl
print("✅ Bibliotecas instaladas. Se aparecer aviso de 'restart', clique em Restart e siga para a próxima célula.")


## 2. Carregar o Dataset Apex (500+ treino + teste) — SEM TOKEN

**Você NÃO precisa de token do Hugging Face.** O dataset da Apex vem direto do seu
repositório (`apex_train.jsonl` = treino, `apex_test.jsonl` = teste separado), e o modelo
base é baixado de um espelho aberto na célula 3. Nada de login, nada de lock-in.


In [ ]:
import os, json

# ═══════════════════════════════════════════════
# DATASET APEX (treino + teste) — SEM TOKEN, SEM LOGIN
# ═══════════════════════════════════════════════
# Tenta baixar do GitHub. Se o repositório for PRIVADO, o download falha e o
# notebook pede o UPLOAD MANUAL dos 2 arquivos (arraste-os na janela que abrir).
REPO_RAW = "https://raw.githubusercontent.com/jedgard70/apex-ai-copilot-platform/main/training_data"

def load_jsonl(fname):
    if not os.path.exists(fname) or os.path.getsize(fname) == 0:
        return []
    with open(fname, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

print("📤 Tentando baixar o dataset da Apex do GitHub...")
os.system(f"wget -q {REPO_RAW}/apex_train.jsonl -O apex_train.jsonl")
os.system(f"wget -q {REPO_RAW}/apex_test.jsonl -O apex_test.jsonl")
train_raw = load_jsonl("apex_train.jsonl")
test_raw = load_jsonl("apex_test.jsonl")

# Fallback (repo privado): upload manual
if not train_raw:
    print("\n⚠️ Download automático falhou (o repositório é privado).")
    print("👉 Faça UPLOAD MANUAL agora dos 2 arquivos que estão em training_data/ no seu PC:")
    print("   • apex_train.jsonl   (845 exemplos)")
    print("   • apex_test.jsonl    (211 exemplos)")
    print("   (arraste os dois na janela de upload que vai abrir)\n")
    from google.colab import files
    files.upload()
    train_raw = load_jsonl("apex_train.jsonl")
    test_raw = load_jsonl("apex_test.jsonl")

assert train_raw, "❌ apex_train.jsonl não carregou. Faça upload do arquivo e rode de novo."
print(f"\n✅ Treino: {len(train_raw)} exemplos | Teste (separado): {len(test_raw)} exemplos")
print("📝 Exemplo de treino:")
print(json.dumps(train_raw[0], ensure_ascii=False, indent=2)[:300])


## 3. Carregar Modelo Gemma 2 2B e Tokenizador

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ═════════════════════════════════════════════════
# BASE ABERTA (SEM TOKEN, SEM LOCK-IN)
# ═════════════════════════════════════════════════
# Usamos um espelho ABERTO do Gemma 2 2B (Unsloth) que NÃO exige token do
# Hugging Face nem aceite de licença. Você não fica preso a ninguém: baixa a
# base uma vez, treina, e o resultado final (GGUF) é 100% seu e roda offline.
#
# Se algum dia quiser usar a base oficial do Google, é só trocar por
# "google/gemma-2-2b-it" e definir HF_TOKEN — mas NÃO é necessário.
MODEL_CANDIDATES = [
    "unsloth/gemma-2-2b-it",       # espelho aberto (sem token)
    "unsloth/gemma-2-2b-it-bnb-4bit",  # espelho aberto já quantizado
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

MODEL_NAME = None
model = None
tokenizer = None
last_err = None
for candidate in MODEL_CANDIDATES:
    try:
        print(f"🔄 Tentando base aberta: {candidate} (sem token)...")
        tokenizer = AutoTokenizer.from_pretrained(candidate)
        tokenizer.padding_side = "right"
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            candidate,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
        MODEL_NAME = candidate
        break
    except Exception as e:
        last_err = e
        print(f"   ⚠️ {candidate} falhou: {str(e)[:120]}")

if model is None:
    raise RuntimeError(
        "Não consegui baixar nenhuma base aberta. Verifique a conexão. "
        f"Último erro: {last_err}"
    )

print(f"\n✅ Base carregada SEM token: {MODEL_NAME}")
print(f"   {sum(p.numel() for p in model.parameters())/1e9:.2f}B parâmetros | Device: {model.device}")
print(f"   Pad: '{tokenizer.pad_token}' id={tokenizer.pad_token_id}")


## 4. Configurar LoRA (Fine-Tuning Eficiente)

LoRA permite fine-tuning com pouquíssima memória ao treinar apenas matrizes de adaptação.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Configuração LoRA
lora_config = LoraConfig(
    r=8,                    # Rank do LoRA (maior = mais params treináveis)
    lora_alpha=32,          # Escala (32 é padrão para rank 8)
    target_modules=[        # Layers onde LoRA será aplicado
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Prepara modelo para treino em quantização 4-bit
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Mostra quantos parâmetros serão treinados
model.print_trainable_parameters()
# Exemplo de output: trainable params: ~8M / total: ~2.2B → só 0.4% treináveis!

## 5. Preparar Dataset no Formato Instruct

In [ ]:
from datasets import Dataset

# Formato instruct do Gemma:
# <start_of_turn>user\n{pergunta}<end_of_turn>\n<start_of_turn>model\n{resposta}<end_of_turn>
def format_example(ex):
    user_text = ex.get("input") or ex.get("messages", [{}])[0].get("content", "")
    model_text = ex.get("output") or (ex.get("messages", [{}, {}])[1].get("content", "") if ex.get("messages") else "")
    return {"text": f"<start_of_turn>user\n{user_text}<end_of_turn>\n<start_of_turn>model\n{model_text}<end_of_turn>"}

train_formatted = [format_example(ex) for ex in train_raw]
train_dataset = Dataset.from_list(train_formatted)

# Pequeno split de validação a partir do TREINO (o apex_test.jsonl fica reservado
# só para a avaliação final na célula 7 — o modelo nunca o vê durante o treino).
split = train_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"✅ Treino: {len(train_dataset)} | Validação interna: {len(eval_dataset)}")
print(f"   Teste final reservado (apex_test.jsonl): {len(test_raw)} exemplos")
print(f"\n📝 Exemplo formatado:\n{train_formatted[0]['text'][:280]}")


## 6. Configurar TrainingArguments e Iniciar Treinamento

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./gemma-apex-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # Batch efetivo = 8
    learning_rate=2e-5,
    warmup_steps=10,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=50,
    save_total_limit=2,
    fp16=True,                        # Ativa mixed precision (essencial na T4)
    bf16=False,
    gradient_checkpointing=True,      # Economiza VRAM
    optim="paged_adamw_8bit",         # Otimizador eficiente para 4-bit
    report_to="none",                 # Não enviar para wandb/tensorboard
    push_to_hub=False,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    ddp_find_unused_parameters=False,
)

def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("🚀 Iniciando treinamento...")
print(f"   Batch efetivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Steps por época: ~{len(tokenized_train) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print(f"   Total de steps: ~{len(tokenized_train) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}")

trainer.train()

## 7. Avaliar no Conjunto de TESTE Separado (perguntas nunca vistas no treino)


In [ ]:
import torch, random

# Salva o modelo fine-tunado (adapters LoRA)
output_dir = "./gemma-apex-finetuned-final"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ Adapters LoRA salvos em {output_dir}")

def apex_generate(prompt, max_new_tokens=160):
    inputs = tokenizer(
        f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n",
        return_tensors="pt", truncation=True, max_length=256,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.7, do_sample=True, top_p=0.95,
                             pad_token_id=tokenizer.pad_token_id)
    resp = tokenizer.decode(out[0], skip_special_tokens=True)
    return resp.split("<start_of_turn>model\n")[-1].strip()

# ═══════════════════════════════════════════════════════════
# AVALIAÇÃO no conjunto de TESTE separado (apex_test.jsonl)
# Mede sobreposição de palavras com a resposta esperada (proxy simples de acerto).
# ═══════════════════════════════════════════════════════════
print("\n🧪 Avaliando no conjunto de teste separado...\n")
sample = test_raw if len(test_raw) <= 30 else random.sample(test_raw, 30)

def overlap_score(pred, ref):
    p = set(pred.lower().split()); r = set(ref.lower().split())
    return len(p & r) / max(1, len(r))

scores = []
for i, ex in enumerate(sample):
    q = ex.get("input", ""); expected = ex.get("output", "")
    ans = apex_generate(q)
    s = overlap_score(ans, expected); scores.append(s)
    if i < 5:  # mostra 5 exemplos
        print(f"🧑 {q}")
        print(f"🤖 {ans[:180]}")
        print(f"   overlap c/ esperado: {s:.0%}\n" + "-" * 50)

if scores:
    print(f"\n📊 Score médio de sobreposição no teste ({len(scores)} exemplos): {sum(scores)/len(scores):.1%}")
    print("   (proxy simples — quanto maior, mais o modelo aprendeu o estilo/conteúdo Apex)")


## 8. Exportar para GGUF Portável (roda LOCAL no .exe, site e apps — sem lock-in)

Aqui o modelo vira um único arquivo **`.gguf`** que roda em qualquer lugar via
**Ollama / llama.cpp**, sem depender de nenhum provedor. O upload para o Hugging Face
é **opcional e PRIVADO** (só se você quiser um backup na nuvem).


In [ ]:
import os

# ═══════════════════════════════════════════════════════════
# PASSO 1: Fundir LoRA no modelo base → modelo completo
# ═══════════════════════════════════════════════════════════
print("🔀 Fundindo adapters LoRA no modelo base...")
merged_model = model.merge_and_unload()
merged_dir = "./gemma-apex-merged"
merged_model.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
print(f"✅ Modelo fundido salvo em {merged_dir}/")

# ═══════════════════════════════════════════════════════════
# PASSO 2: Converter para GGUF (portável — llama.cpp/Ollama)
# ═══════════════════════════════════════════════════════════
print("\n🔧 Convertendo para GGUF (formato portável, sem lock-in)...")
if not os.path.exists("llama.cpp"):
    os.system("git clone --depth 1 https://github.com/ggerganov/llama.cpp")
os.system("pip install -q -r llama.cpp/requirements.txt gguf sentencepiece protobuf")

GGUF_FP16 = "apex-ai-f16.gguf"
GGUF_Q4 = "apex-ai.gguf"   # quantizado Q4_K_M — leve, roda em CPU

print("   HF → GGUF (fp16)...")
os.system(f"python llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile {GGUF_FP16} --outtype f16")

print("   Quantizando para Q4_K_M (leve, roda em CPU)...")
os.system("cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF >/dev/null 2>&1")
os.system("cmake --build llama.cpp/build --target llama-quantize -j 4 >/dev/null 2>&1")
quantize_bin = "llama.cpp/build/bin/llama-quantize"
if not os.path.exists(quantize_bin):
    quantize_bin = "llama.cpp/build/llama-quantize"
os.system(f"{quantize_bin} {GGUF_FP16} {GGUF_Q4} Q4_K_M")

if os.path.exists(GGUF_Q4):
    print(f"✅ GGUF final: {GGUF_Q4} ({os.path.getsize(GGUF_Q4)/1e6:.0f} MB) — SEU modelo portável")
else:
    print("⚠️ Quantização não gerou o arquivo. Usando fp16:", GGUF_FP16)
    GGUF_Q4 = GGUF_FP16 if os.path.exists(GGUF_FP16) else None

# ═══════════════════════════════════════════════════════════
# PASSO 3: Modelfile do Ollama (rodar LOCAL no .exe / site / apps)
# ═══════════════════════════════════════════════════════════
modelfile = f'''FROM ./{os.path.basename(GGUF_Q4) if GGUF_Q4 else GGUF_FP16}
TEMPLATE """<start_of_turn>user
{{{{ .Prompt }}}}<end_of_turn>
<start_of_turn>model
"""
SYSTEM "Você é a Apex AI, plataforma profissional de arquitetura, construção, BIM, orçamentos, marketing e gestão. Responda em português, de forma técnica e direta, sem inventar dados ou integrações que não existem."
PARAMETER temperature 0.7
PARAMETER top_p 0.95
PARAMETER stop "<end_of_turn>"
'''
with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)
print("\n📄 Modelfile do Ollama criado.")

# ═══════════════════════════════════════════════════════════
# PASSO 4: Baixar o GGUF + Modelfile (o modelo é SEU, roda local)
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📥 COMO USAR LOCAL (sem depender de ninguém):")
print("=" * 60)
print(f"""
1. Baixe pelo painel do Colab (Files → download):
     • {GGUF_Q4 or GGUF_FP16}   ← seu modelo Apex
     • Modelfile

2. No seu PC/servidor, instale o Ollama: https://ollama.com

3. Crie o modelo Apex local:
     ollama create apex-ai -f Modelfile

4. Rode sem API key:
     ollama run apex-ai "Quem é você?"

5. Sirva para o site e apps (API local compatível):
     POST http://localhost:11434/api/chat
     {{ "model": "apex-ai", "messages": [{{"role":"user","content":"O que é a Apex?"}}] }}

   → No Apex Desktop (.exe) o Ollama roda embarcado.
   → No site/apps, aponte o provedor "apex-local" para o endpoint do Ollama.
""")
try:
    from google.colab import files as _f
    if GGUF_Q4 and os.path.exists(GGUF_Q4):
        _f.download(GGUF_Q4)
    _f.download("Modelfile")
except Exception as e:
    print("(Baixe manualmente pelo painel Files do Colab.)", e)

# ═══════════════════════════════════════════════════════════
# PASSO 5 (OPCIONAL): backup PRIVADO no Hugging Face — DESLIGADO por padrão
# ═══════════════════════════════════════════════════════════
# Deixe False para NÃO usar HuggingFace de forma alguma. O modelo já é 100% seu
# e local (GGUF). Só ligue se um dia quiser um backup privado na nuvem.
UPLOAD_PRIVADO_HF = False
if UPLOAD_PRIVADO_HF and GGUF_Q4:
    from getpass import getpass
    from huggingface_hub import HfApi, create_repo, login as hf_login
    _tok = getpass("Token HF 'write' (só para backup privado opcional): ")
    hf_login(token=_tok, add_to_git_credential=False)
    repo_id = "jedgard70/apex-ai-gguf"
    create_repo(repo_id, private=True, exist_ok=True, token=_tok)   # PRIVADO
    HfApi(token=_tok).upload_file(
        path_or_fileobj=GGUF_Q4, path_in_repo=os.path.basename(GGUF_Q4), repo_id=repo_id)
    print(f"☁️ Backup PRIVADO enviado: https://huggingface.co/{repo_id} (só você acessa)")
else:
    print("ℹ️ HuggingFace NÃO usado. O modelo é 100% seu e local (GGUF). Zero lock-in.")

print("\n" + "=" * 60)
print("🎉 PRONTO! Modelo Apex treinado e exportado em GGUF portável.")
print("=" * 60)
